# Chapter 4 - Data Preprocessing

In the previous chapter we learned how to bring data into Python. But raw data
is rarely ready for analysis or Machine Learning. It may contain missing values,
wrong data types, duplicate rows, inconsistent entries, or untidy structure.

In this chapter we will clean a clinical trial dataset step by step. The aim is
to understand how data assessment and data cleaning work in a real project.

## Dirty Data and Messy Data

Before cleaning any dataset, we first need to understand what kind of problem we
are fixing.

**Dirty data** means data quality problems. Examples include missing values,
duplicate records, incorrect data types, impossible values, inconsistent entries,
and corrupted text.

**Messy data** means data structure problems. Examples include multiple values
stored in one column, one variable spread across many columns, or different
observational units mixed into the same table.

In short, dirty data needs to be corrected, and messy data needs to be organized.

## Practical Data Cleaning

We will work with a clinical trial dataset that compares Ayurvedic and
Allopathic treatments for managing blood sugar levels.

The data is available in two CSV files:

- `patients.csv`: patient demographic details
- `treatments.csv`: treatment details and blood sugar readings

We will first assess both datasets, then clean them using the issues we find.

## Step 1 - Data Assessment

Data assessment means examining the dataset before cleaning it. This helps us
understand what is wrong and decide which cleaning steps are required.

Assessment can be done in two ways:

- **Manual assessment**: visually inspecting the data by opening it in a spreadsheet
- **Programmatic assessment**: using code to check the data systematically

### Manual Assessment

By manually inspecting the files, we can identify the following issues.

**In `patients.csv`:**

- missing values in `weight_kg`, `height_cm`, and `contact_info`
- inconsistent gender entries such as `Male`, `male`, and `M`
- inconsistent state entries such as full names and abbreviations
- special characters in some names
- `contact_info` stores email and phone number together

**In `treatments.csv`:**

- treatment dosage is spread across two columns
- dash `-` is used where a patient did not receive a treatment
- dosage values contain the unit `mg`
- dosage range is stored as one value, such as `250mg-500mg`

### Programmatic Assessment - Loading patients.csv

Now we use Pandas to inspect the dataset with code. We load the local CSV file
so that the notebook can run without depending on the internet.

In [272]:
import pandas as pd

patients_path = 'https://raw.githubusercontent.com/kanetkar/LULML/refs/heads/main/ch04/patients.csv'

# Load the patient dataset into a DataFrame.
patients = pd.read_csv(patients_path)

print("patients.csv information:")
patients.info()

patients.csv information:
<class 'pandas.DataFrame'>
RangeIndex: 405 entries, 0 to 404
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   patient_id    405 non-null    int64  
 1   first_name    405 non-null    str    
 2   last_name     405 non-null    str    
 3   birthdate     405 non-null    str    
 4   gender        405 non-null    str    
 5   weight_kg     400 non-null    float64
 6   height_cm     400 non-null    float64
 7   contact_info  359 non-null    str    
 8   address       405 non-null    str    
 9   city          405 non-null    str    
 10  state         405 non-null    str    
 11  pin_code      405 non-null    int64  
 12  country       405 non-null    str    
dtypes: float64(2), int64(2), str(9)
memory usage: 41.3 KB


The `info()` output helps us check column names, non-null counts, and data
types. It shows that `birthdate` is stored as text and that some columns have
missing values.

In [273]:
print("Missing values in patients.csv:")
print(patients.isnull().sum())

Missing values in patients.csv:
patient_id       0
first_name       0
last_name        0
birthdate        0
gender           0
weight_kg        5
height_cm        5
contact_info    46
address          0
city             0
state            0
pin_code         0
country          0
dtype: int64


The output shows missing values in `weight_kg`, `height_cm`, and `contact_info`.
These columns will need cleaning before the data can be used reliably.

In [274]:
print("Duplicate rows in patients.csv:")
print(patients.duplicated().sum())

Duplicate rows in patients.csv:
5


Duplicate rows can overrepresent some patients. So, if duplicates are present,
we should remove them during cleaning.

In [275]:
print("Descriptive statistics for patients.csv:")
print(patients.describe())

Descriptive statistics for patients.csv:
       patient_id   weight_kg   height_cm       pin_code
count  405.000000  400.000000  400.000000     405.000000
mean   200.200000   68.769360  168.734845  578235.380247
std    115.803529   16.915796   22.609489  248240.704729
min      1.000000    5.000000  136.037446  104666.000000
25%    100.000000   58.378059  160.041690  375097.000000
50%    201.000000   69.166507  166.970772  579776.000000
75%    300.000000   79.604325  173.356025  789545.000000
max    400.000000  116.183212  350.000000  999695.000000


The descriptive statistics reveal implausible values. For example, a weight of
5 kg and a height of 350 cm are not realistic adult patient values. These are
likely data entry errors.

### Programmatic Assessment - Loading treatments.csv

Next, we inspect the treatment details in the same way.

In [276]:
treatments_path = 'https://raw.githubusercontent.com/kanetkar/LULML/refs/heads/main/ch04/treatments.csv'

# Load the treatments dataset into a DataFrame.
treatments = pd.read_csv(treatments_path)

print("treatments.csv information:")
treatments.info()

treatments.csv information:
<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 7 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   patient_id                  400 non-null    int64  
 1   ayurvedic_medicine_dosage   400 non-null    str    
 2   allopathic_medicine_dosage  400 non-null    str    
 3   fasting_blood_sugar_start   400 non-null    float64
 4   fasting_blood_sugar_end     400 non-null    float64
 5   bs_change                   390 non-null    float64
 6   adverse_event               164 non-null    str    
dtypes: float64(3), int64(1), str(3)
memory usage: 22.0 KB


The dosage columns are stored as text because they contain ranges, units, and
dash values. We will clean these columns later.

In [277]:
print("Missing values in treatments.csv:")
print(treatments.isnull().sum())

Missing values in treatments.csv:
patient_id                      0
ayurvedic_medicine_dosage       0
allopathic_medicine_dosage      0
fasting_blood_sugar_start       0
fasting_blood_sugar_end         0
bs_change                      10
adverse_event                 236
dtype: int64


`bs_change` has missing values. The `adverse_event` column has many missing
values, which may mean that those patients did not experience side effects.

In [278]:
print("Duplicate rows in treatments.csv:")
print(treatments.duplicated().sum())

Duplicate rows in treatments.csv:
0


In [279]:
print("Descriptive statistics for treatments.csv:")
print(treatments.describe())

Descriptive statistics for treatments.csv:
       patient_id  fasting_blood_sugar_start  fasting_blood_sugar_end  \
count  400.000000                 400.000000               400.000000   
mean   200.500000                 151.303500               121.265000   
std    115.614301                  27.468836                30.065761   
min      1.000000                 100.100000                56.800000   
25%    100.750000                 129.700000               100.275000   
50%    200.500000                 152.100000               123.100000   
75%    300.250000                 174.425000               144.575000   
max    400.000000                 199.300000               186.800000   

        bs_change  
count  390.000000  
mean   -29.854359  
std     11.212785  
min    -49.900000  
25%    -38.975000  
50%    -30.250000  
75%    -20.400000  
max    -10.000000  


## Step 2 - Data Cleaning

Data cleaning means transforming raw data into a consistent and usable form.
We will clean `patients.csv` first and then clean `treatments.csv`.

### Cleaning patients.csv - Handling Missing Values

For numeric columns such as `weight_kg` and `height_cm`, we fill missing values
with the column mean. 

For `contact_info`, we use a clear placeholder.

In [280]:
print("Before handling missing values:")
print(patients[["weight_kg", "height_cm", "contact_info"]].isnull().sum())

# Fill missing weight values with the average weight.
mean_weight = patients["weight_kg"].mean()
patients["weight_kg"] = patients["weight_kg"].fillna(mean_weight)

# Fill missing height values with the average height.
mean_height = patients["height_cm"].mean()
patients["height_cm"] = patients["height_cm"].fillna(mean_height)

# Missing contact details are replaced with a readable placeholder.
patients["contact_info"] = patients["contact_info"].fillna("Not Provided")

print("\nAfter handling missing values:")
print(patients[["weight_kg", "height_cm", "contact_info"]].isnull().sum())

Before handling missing values:
weight_kg        5
height_cm        5
contact_info    46
dtype: int64

After handling missing values:
weight_kg       0
height_cm       0
contact_info    0
dtype: int64


The verification output confirms that these three columns no longer contain
missing values.

### Cleaning patients.csv - Standardising Inconsistent Entries

The same category should be written in one consistent way. 

For example, `Male`, `male`, and `M` should all become `Male`.

In [281]:
gender_dict = {
    "male": "Male",
    "M": "Male",
    "Male": "Male",
    "female": "Female",
    "F": "Female",
    "Female": "Female",
}

print("Before standardising gender values:")
print(patients["gender"].value_counts(dropna=False))

patients["gender"] = patients["gender"].map(gender_dict)

print("\nAfter standardising gender values:")
print(patients["gender"].value_counts(dropna=False))

Before standardising gender values:
gender
male      83
F         73
Male      66
Female    66
female    60
M         57
Name: count, dtype: int64

After standardising gender values:
gender
Male      206
Female    199
Name: count, dtype: int64


Only two standard gender values remain, which makes the column easier to
analyze later.

In [282]:
# State names are also standardized so abbreviations and full names match.
state_mapping = {
    "Maharashtra": "Maharashtra",
    "MH": "Maharashtra",
    "Karnataka": "Karnataka",
    "KA": "Karnataka",
    "Tamil Nadu": "Tamil Nadu",
    "TN": "Tamil Nadu",
    "West Bengal": "West Bengal",
    "WB": "West Bengal",
    "Gujarat": "Gujarat",
    "GJ": "Gujarat",
}

print("Before standardising state values:")
print(patients["state"].value_counts(dropna=False).sort_index())

patients["state"] = patients["state"].map(state_mapping)

print("\nAfter standardising state values:")
print(patients["state"].value_counts(dropna=False).sort_index())

Before standardising state values:
state
GJ             36
Gujarat        40
KA             34
Karnataka      45
MH             51
Maharashtra    33
TN             39
Tamil Nadu     52
WB             46
West Bengal    29
Name: count, dtype: int64

After standardising state values:
state
Gujarat        76
Karnataka      79
Maharashtra    84
Tamil Nadu     91
West Bengal    75
Name: count, dtype: int64


Standardized state names prevent incorrect grouping during future analysis.

### Cleaning patients.csv - Removing Special Characters

Some last names contain special characters. We keep only letters and spaces.

We have used a regular expression `r"[^a-zA-Z\s]"` to remove any character that is not a letter or space.

In [283]:
print("Before removing special characters from last_name:")
print(patients.loc[patients["last_name"].str.contains(r"[^a-zA-Z\s]", na=False), "last_name"].head())

patients["last_name"] = patients["last_name"].str.replace(r"[^a-zA-Z\s]", "", regex=True)

print("\nAfter removing special characters from last_name:")
print(patients.loc[patients["last_name"].str.contains(r"[^a-zA-Z\s]", na=False), "last_name"].head())
print("Special characters remaining:", patients["last_name"].str.contains(r"[^a-zA-Z\s]").sum())

Before removing special characters from last_name:
6      Mehta*
26     Singh*
94       Das*
129    Kumar*
145      Das*
Name: last_name, dtype: str

After removing special characters from last_name:
Series([], Name: last_name, dtype: str)
Special characters remaining: 0


A count of zero confirms that the special characters have been removed.

### Cleaning patients.csv - Fixing Data Types

`birthdate` should be a datetime column, and `pin_code` should be text. 

Pin codes are identifiers, not numbers - so keeping them as text is safer.

In [284]:
print("Before fixing data types:")
print(patients[["birthdate", "pin_code"]].dtypes)
print(patients[["birthdate", "pin_code"]].head())

patients["birthdate"] = pd.to_datetime(patients["birthdate"], errors="coerce")
patients["pin_code"] = patients["pin_code"].astype(str).str.zfill(6)

print("\nAfter fixing data types:")
print(patients[["birthdate", "pin_code"]].dtypes)
print(patients[["birthdate", "pin_code"]].head())

Before fixing data types:
birthdate      str
pin_code     int64
dtype: object
             birthdate  pin_code
0  1990-12-22 11:45:42    943135
1  1980-05-11 05:33:39    445549
2  2001-05-05 03:07:08    561607
3  1964-12-18 14:13:34    450996
4  1971-07-29 11:32:58    551533

After fixing data types:
birthdate    datetime64[us]
pin_code                str
dtype: object
            birthdate pin_code
0 1990-12-22 11:45:42   943135
1 1980-05-11 05:33:39   445549
2 2001-05-05 03:07:08   561607
3 1964-12-18 14:13:34   450996
4 1971-07-29 11:32:58   551533


The data types now match the meaning of the columns.

### Cleaning patients.csv - Removing Duplicates

We remove duplicate patient records using `patient_id`, keeping the first
occurrence of each patient.

In [285]:
print("Before removing duplicate patient records:")
print("Duplicate patient_id count:", patients.duplicated(subset="patient_id").sum())
print(patients.loc[patients.duplicated(subset="patient_id", keep=False), ["patient_id", "first_name", "last_name"]].head())

patients = patients.drop_duplicates(subset="patient_id", keep="first")

print("\nAfter removing duplicate patient records:")
print("Duplicate patient_id count:", patients.duplicated(subset="patient_id").sum())

Before removing duplicate patient records:
Duplicate patient_id count: 5
     patient_id first_name last_name
3             4       Arin     Mehta
52           53        Ish    Chopra
203         204        Dua     Patel
299         300       Aksh     Kumar
319         320        Dua     Kumar

After removing duplicate patient records:
Duplicate patient_id count: 0


The duplicate count is now zero.

### Cleaning patients.csv - Capping Implausible Values

Instead of leaving impossible values in the data, we cap them to reasonable
limits.

In [286]:
print("Before capping implausible values:")
print(patients.loc[(patients["weight_kg"] < 30) | (patients["height_cm"] > 250), ["patient_id", "weight_kg", "height_cm"]].head())
print("\nMinimum weight before capping:", patients["weight_kg"].min())
print("Maximum height before capping:", patients["height_cm"].max())

min_weight = 30
patients.loc[patients["weight_kg"] < min_weight, "weight_kg"] = min_weight

max_height = 250
patients.loc[patients["height_cm"] > max_height, "height_cm"] = max_height

print("\nAfter capping implausible values:")
print(patients.loc[(patients["weight_kg"] < 30) | (patients["height_cm"] > 250), ["patient_id", "weight_kg", "height_cm"]].head())
print("\nMinimum weight after capping:", patients["weight_kg"].min())
print("Maximum height after capping:", patients["height_cm"].max())

Before capping implausible values:
     patient_id  weight_kg   height_cm
60           61   5.000000  151.555495
127         128   5.000000  166.467137
194         195  88.018209  350.000000
213         214   5.000000  164.522886
226         227  65.951876  350.000000

Minimum weight before capping: 5.0
Maximum height before capping: 350.0

After capping implausible values:
Empty DataFrame
Columns: [patient_id, weight_kg, height_cm]
Index: []

Minimum weight after capping: 30.0
Maximum height after capping: 250.0


The extreme values have been brought into a plausible range.

### Cleaning patients.csv - Tidying Contact Information

`contact_info` contains two pieces of information in one column: email and
phone number. We split it into two separate columns.

In [287]:
print("Before splitting contact_info:")
print(patients[["contact_info"]].head())

contact_split = patients["contact_info"].str.split(";", expand=True)
patients["email"] = contact_split[0]
patients["phone_number"] = contact_split[1]

print("\nAfter splitting contact_info:")
print(patients[["email", "phone_number"]].head())

Before splitting contact_info:
                            contact_info
0    dua.mehta@example.in;+91-7277121963
1  kiran.gupta@example.in;+91-9694387705
2   ashi.mehta@example.in;+91-8527696207
3   arin.mehta@example.in;+91-7841810119
4   arin.kumar@example.in;+91-7311598149

After splitting contact_info:
                    email    phone_number
0    dua.mehta@example.in  +91-7277121963
1  kiran.gupta@example.in  +91-9694387705
2   ashi.mehta@example.in  +91-8527696207
3   arin.mehta@example.in  +91-7841810119
4   arin.kumar@example.in  +91-7311598149


In [288]:
# dropping the "contact_info" column as we have already converted all its information to two separate "email" and "phone_number" columns
patients = patients.drop(columns="contact_info")

print("Columns after dropping contact_info:")
print(patients.columns.tolist())

Columns after dropping contact_info:
['patient_id', 'first_name', 'last_name', 'birthdate', 'gender', 'weight_kg', 'height_cm', 'address', 'city', 'state', 'pin_code', 'country', 'email', 'phone_number']


The contact details are now tidier because email and phone number are stored as
separate variables.

### Cleaning treatments.csv - Reshaping from Wide to Long Format

The treatment dosage is currently spread across two columns. 

We use `melt()` to convert those two columns into one `treatment_type` column and one `dosage_range`
column.

In [289]:
print("Before reshaping treatment dosage columns:")
print(treatments[["ayurvedic_medicine_dosage", "allopathic_medicine_dosage"]].head())
print("Shape before melt:", treatments.shape)

treatments_long = treatments.melt(
    id_vars=[
        "patient_id",
        "fasting_blood_sugar_start",
        "fasting_blood_sugar_end",
        "bs_change",
        "adverse_event",
    ],
    value_vars=["ayurvedic_medicine_dosage", "allopathic_medicine_dosage"],
    var_name="treatment_type",
    value_name="dosage_range",
)

print("\nAfter reshaping treatment dosage columns:")
print(treatments_long[["treatment_type", "dosage_range"]].head())
print("Shape after melt:", treatments_long.shape)

Before reshaping treatment dosage columns:
  ayurvedic_medicine_dosage allopathic_medicine_dosage
0                   2mg-3mg                          -
1                         -                500mg-750mg
2                   2mg-2mg                          -
3                   2mg-3mg                          -
4                         -                500mg-750mg
Shape before melt: (400, 7)

After reshaping treatment dosage columns:
              treatment_type dosage_range
0  ayurvedic_medicine_dosage      2mg-3mg
1  ayurvedic_medicine_dosage            -
2  ayurvedic_medicine_dosage      2mg-2mg
3  ayurvedic_medicine_dosage      2mg-3mg
4  ayurvedic_medicine_dosage            -
Shape after melt: (800, 7)


The two dosage columns have been unpivoted into rows. This is a tidier structure
because treatment type is now stored as a value in one column.

In [290]:
print("Before simplifying treatment type names:")
print(treatments_long["treatment_type"].value_counts())

treatments_long["treatment_type"] = (
    treatments_long["treatment_type"]
    .str.replace("_medicine_dosage", "", regex=False)
    .str.capitalize()
)

print("\nAfter simplifying treatment type names:")
print(treatments_long["treatment_type"].value_counts())

Before simplifying treatment type names:
treatment_type
ayurvedic_medicine_dosage     400
allopathic_medicine_dosage    400
Name: count, dtype: int64

After simplifying treatment type names:
treatment_type
Ayurvedic     400
Allopathic    400
Name: count, dtype: int64


Now the names of these columns are much more simple than before

### Cleaning treatments.csv - Removing No-Treatment Rows

A dash means that the patient did not receive that treatment. Those rows are
removed because they are not real dosage records.

In [291]:
print("Before removing no-treatment rows:")
print(treatments_long["dosage_range"].value_counts().head())
print("Shape before removing no-treatment rows:", treatments_long.shape)

treatments_long = treatments_long[treatments_long["dosage_range"] != "-"]

Before removing no-treatment rows:
dosage_range
-              400
250mg-250mg     61
1mg-2mg         54
1mg-1mg         51
500mg-500mg     50
Name: count, dtype: int64
Shape before removing no-treatment rows: (800, 7)


The above output clearly shows us that there are 400 rows which have "-" as input value.

In [292]:
print("\nAfter removing no-treatment rows:")
print(treatments_long[["treatment_type", "dosage_range"]].head())
print("Shape after removing no-treatment rows:", treatments_long.shape)


After removing no-treatment rows:
  treatment_type dosage_range
0      Ayurvedic      2mg-3mg
2      Ayurvedic      2mg-2mg
3      Ayurvedic      2mg-3mg
6      Ayurvedic      2mg-3mg
7      Ayurvedic      1mg-2mg
Shape after removing no-treatment rows: (400, 7)


After removing 400 dash rows, each remaining row represents an actual treatment
received by a patient.

### Cleaning treatments.csv - Splitting Dosage Range

The dosage range stores two values in one column. We split it into starting and
ending dosage columns.

In [293]:
print("Before splitting dosage_range:")
print(treatments_long[["dosage_range"]].head())

treatments_long["dosage_start"] = treatments_long["dosage_range"].str.split("-").str.get(0)
treatments_long["dosage_end"] = treatments_long["dosage_range"].str.split("-").str.get(1)

print("\nAfter splitting dosage_range:")
print(treatments_long[["dosage_start", "dosage_end"]].head())

Before splitting dosage_range:
  dosage_range
0      2mg-3mg
2      2mg-2mg
3      2mg-3mg
6      2mg-3mg
7      1mg-2mg

After splitting dosage_range:
  dosage_start dosage_end
0          2mg        3mg
2          2mg        2mg
3          2mg        3mg
6          2mg        3mg
7          1mg        2mg


In [294]:
treatments_long = treatments_long.drop(columns="dosage_range")

print("Before removing mg unit:")
print(treatments_long[["dosage_start", "dosage_end"]].head())

# Remove the unit from both dosage columns.
treatments_long["dosage_start"] = (
    treatments_long["dosage_start"].str.replace("mg", "", regex=False).str.strip()
)
treatments_long["dosage_end"] = (
    treatments_long["dosage_end"].str.replace("mg", "", regex=False).str.strip()
)

print("\nAfter removing mg unit:")
print(treatments_long[["dosage_start", "dosage_end"]].head())

Before removing mg unit:
  dosage_start dosage_end
0          2mg        3mg
2          2mg        2mg
3          2mg        3mg
6          2mg        3mg
7          1mg        2mg

After removing mg unit:
  dosage_start dosage_end
0            2          3
2            2          2
3            2          3
6            2          3
7            1          2


In [295]:
print("Before numeric conversion:")
print(treatments_long[["dosage_start", "dosage_end"]].dtypes)

# Convert dosage values from text to numbers.
treatments_long["dosage_start"] = pd.to_numeric(treatments_long["dosage_start"], errors="coerce")
treatments_long["dosage_end"] = pd.to_numeric(treatments_long["dosage_end"], errors="coerce")

print("\nAfter numeric conversion:")
print(treatments_long[["dosage_start", "dosage_end"]].dtypes)

Before numeric conversion:
dosage_start    object
dosage_end      object
dtype: object

After numeric conversion:
dosage_start    int64
dosage_end      int64
dtype: object


The dosage values are now numeric, so they can be used for analysis or modeling.

### Cleaning treatments.csv - Handling Missing Values

For missing `bs_change`, we calculate the difference between ending and starting
blood sugar levels. 

For missing `adverse_event`, we fill `None`, assuming no
adverse event was reported.

In [296]:
print("Before handling missing values for bs_change, adverse_event columns:")
print(treatments_long[["bs_change", "adverse_event"]].isnull().sum())

treatments_long["bs_change"] = (
    treatments_long["fasting_blood_sugar_end"] - treatments_long["fasting_blood_sugar_start"]
)

treatments_long["adverse_event"] = treatments_long["adverse_event"].fillna("None")

print("\nAfter handling missing values for bs_change, adverse_event columns:")
print(treatments_long[["bs_change", "adverse_event"]].isnull().sum())

Before handling missing values for bs_change, adverse_event columns:
bs_change         10
adverse_event    236
dtype: int64

After handling missing values for bs_change, adverse_event columns:
bs_change        0
adverse_event    0
dtype: int64


Both columns now have zero missing values.

### Saving the Cleaned Datasets

After cleaning both datasets, we save the cleaned versions as new CSV files.

In [299]:
from pathlib import Path

BASE_DIR = Path.cwd()
if not (BASE_DIR / "patients.csv").exists() and (BASE_DIR / "ch04" / "patients.csv").exists():
    BASE_DIR = BASE_DIR / "ch04"

patients_cleaned_path = BASE_DIR / "patients_cleaned.csv"
treatments_cleaned_path = BASE_DIR / "treatments_cleaned.csv"

patients.to_csv(patients_cleaned_path, index=False)
treatments_long.to_csv(treatments_cleaned_path, index=False)

print("Saved cleaned patients data to:", patients_cleaned_path)
print("Saved cleaned treatments data to:", treatments_cleaned_path)

Saved cleaned patients data to: d:\LULML\ch04\patients_cleaned.csv
Saved cleaned treatments data to: d:\LULML\ch04\treatments_cleaned.csv


## Chapter Recap

In this chapter we cleaned a clinical trial dataset by fixing both dirty-data
and messy-data problems.

- We assessed the data manually and programmatically.
- We handled missing values.
- We standardized inconsistent gender and state values.
- We removed special characters from names.
- We fixed incorrect data types.
- We removed duplicate patient records.
- We capped implausible weight and height values.
- We split combined contact information into separate columns.
- We reshaped treatment dosage data from wide format to long format.
- We split dosage ranges, removed units, and converted dosage values to numbers.
- We saved the cleaned datasets for the next stage of the ML workflow.

The data is now cleaner, more consistent, and ready for Exploratory Data
Analysis.